24BAD071            BUSSINESS INTELLIGENCE              Mithunjai.E

# Assignment 3 — Gemini-Assisted Notebook
## Clean, Transform, Analyze: Pandas Analytic Functions

**Dataset:** Retail Sales Dataset (Region × Category, 2024)  
**Tools:** Google Colab + Gemini AI Assistant  
**Student:** Mithunjai.E | AI & Data Science, KCT Coimbatore

---

## Task 1 — Load the Messy Dataset

We use a retail sales dataset with **deliberately introduced data quality issues**:
- **Missing values** in `Sales_Amount` (rows 1006, 1015)
- **Outlier** in `Sales_Amount` (row 1011 — ₹98,000 vs. typical ₹300–2,600)
- **Inconsistent date formats**: `YYYY-MM-DD`, `DD/MM/YYYY`, `DD-MM-YYYY`
- **Inconsistent Region casing**: `south`, `EAST` vs `North`, `West`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Load the dataset ──
# In Colab, upload sales_messy.csv using the Files panel (left sidebar)
# OR paste the raw data directly via io.StringIO below:

from io import StringIO

raw_data = """Order_ID,Region,Category,Sales_Amount,Order_Date,Sales_Rep
1001,North,Electronics,1500.00,2024-01-15,Alice
1002,South,Furniture,850.50,15/02/2024,Bob
1003,East,Electronics,2200.00,2024-03-10,Carol
1004,West,Clothing,430.00,2024-01-22,Dave
1005,North,Furniture,920.00,2024-02-18,Alice
1006,South,Electronics,,2024-03-05,Bob
1007,East,Clothing,310.00,10-04-2024,Carol
1008,West,Electronics,1750.00,2024-02-28,Dave
1009,North,Clothing,275.00,2024-04-12,Alice
1010,South,Furniture,1100.00,2024-01-30,Bob
1011,East,Electronics,98000.00,2024-03-22,Carol
1012,West,Furniture,640.00,2024-04-05,Dave
1013,North,Electronics,1320.00,2024-05-10,Alice
1014,south,Clothing,490.00,2024-05-15,Bob
1015,East,Furniture,,2024-02-20,Carol
1016,West,Clothing,380.00,2024-06-01,Dave
1017,North,Furniture,870.00,2024-06-14,Alice
1018,South,Electronics,2050.00,2024-05-28,Bob
1019,East,Clothing,420.00,2024-06-10,Carol
1020,West,Electronics,1900.00,2024-04-22,Dave
1021,North,Clothing,310.00,2024-07-03,Alice
1022,South,Furniture,780.00,03/07/2024,Bob
1023,East,Electronics,2400.00,2024-07-18,Carol
1024,West,Furniture,560.00,2024-08-01,Dave
1025,North,Electronics,1650.00,2024-08-12,Alice
1026,South,Clothing,520.00,2024-08-25,Bob
1027,EAST,Furniture,730.00,2024-09-05,Carol
1028,West,Clothing,295.00,2024-09-18,Dave
1029,North,Furniture,990.00,2024-09-30,Alice
1030,South,Electronics,2300.00,2024-10-10,Bob
1031,East,Clothing,360.00,2024-10-22,Carol
1032,West,Electronics,2100.00,2024-11-05,Dave
1033,North,Clothing,240.00,2024-11-18,Alice
1034,South,Furniture,840.00,2024-11-30,Bob
1035,East,Electronics,2600.00,2024-12-10,Carol
1036,West,Furniture,470.00,2024-12-22,Dave"""

df_raw = pd.read_csv(StringIO(raw_data))
print(f"Shape: {df_raw.shape}")
print(f"\nColumn dtypes:\n{df_raw.dtypes}")
df_raw.head(12)

In [ ]:
# Quick audit of data quality issues
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

print(f"\n1. Missing values per column:")
print(df_raw.isnull().sum())

print(f"\n2. Unique Region values (check casing):")
print(df_raw['Region'].unique())

print(f"\n3. Sample Order_Date values (check format inconsistency):")
print(df_raw['Order_Date'].sample(8, random_state=42).tolist())

print(f"\n4. Sales_Amount statistics (check outlier):")
print(df_raw['Sales_Amount'].describe())

## Task 2 — Clean the Dataset

### 2(a) Handle Missing Values

**Strategy:**  
`Sales_Amount` is a numeric revenue field. We cannot fabricate revenue, so we do NOT drop the rows (that loses order context). Instead, we **impute with the median** of the same Region × Category group — a far more defensible estimate than the global median, because electronics in the East behave differently from clothing in the West.

> **Gemini suggestion received:** `df['Sales_Amount'].fillna(df['Sales_Amount'].mean())` — global mean imputation.  
> **My correction:** Replaced with grouped-median imputation, since the global mean is pulled up by the outlier (see 2b) and ignores product/region context.


In [ ]:
df = df_raw.copy()

# ── 2(a) Missing value imputation: group median ──
def fill_group_median(group):
    median = group['Sales_Amount'].median()
    group['Sales_Amount'] = group['Sales_Amount'].fillna(median)
    return group

df = df.groupby(['Region', 'Category'], group_keys=False).apply(fill_group_median)

# Fallback: if a group had ALL nulls (rare), use global median
global_median = df['Sales_Amount'].median()
df['Sales_Amount'] = df['Sales_Amount'].fillna(global_median)

print("Missing values after imputation:", df['Sales_Amount'].isnull().sum())
print(f"Values filled with group median — no rows dropped.")
df[df_raw['Sales_Amount'].isnull()][['Order_ID','Region','Category','Sales_Amount']]

### 2(b) Detect and Handle the Outlier

**Method: IQR (Interquartile Range)**  
IQR is robust to extreme values and does not assume a normal distribution — appropriate for sales data.  

- Q1 = 25th percentile, Q3 = 75th percentile  
- IQR = Q3 − Q1  
- Outlier threshold: values above Q3 + 1.5×IQR  

**Decision:** Cap outliers at the upper fence (Winsorization) rather than dropping — the order is real, but the amount is likely a data-entry error (₹98,000 instead of ₹980 or ₹9,800).


In [ ]:
# ── 2(b) Outlier detection via IQR ──
Q1 = df['Sales_Amount'].quantile(0.25)
Q3 = df['Sales_Amount'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
lower_fence = Q1 - 1.5 * IQR

print(f"Q1 = {Q1:.2f} | Q3 = {Q3:.2f} | IQR = {IQR:.2f}")
print(f"Lower fence = {lower_fence:.2f} | Upper fence = {upper_fence:.2f}")

outliers = df[df['Sales_Amount'] > upper_fence]
print(f"\nOutliers detected:")
print(outliers[['Order_ID', 'Region', 'Category', 'Sales_Amount']])

# ── Winsorize: cap at upper fence ──
df['Sales_Amount'] = df['Sales_Amount'].clip(upper=upper_fence)
print(f"\nAfter capping — max Sales_Amount: {df['Sales_Amount'].max():.2f}")

In [ ]:
# ── Visual confirmation ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(df_raw['Sales_Amount'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='#ffcccc'))
axes[0].set_title('Before: Raw Sales_Amount')
axes[0].set_ylabel('Amount (₹)')

axes[1].boxplot(df['Sales_Amount'], patch_artist=True,
                boxprops=dict(facecolor='#ccffcc'))
axes[1].set_title('After: Cleaned Sales_Amount')
axes[1].set_ylabel('Amount (₹)')

plt.suptitle('Outlier Handling — IQR Winsorization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2(c) Standardize Inconsistent Formatting

Two issues to fix:
1. **`Order_Date`** has three formats: `YYYY-MM-DD`, `DD/MM/YYYY`, `DD-MM-YYYY` → parse all to a single `datetime`, then extract `Quarter`.
2. **`Region`** has casing inconsistencies: `south`, `EAST` → title-case everything.


In [ ]:
# ── 2(c) Fix Region casing ──
df['Region'] = df['Region'].str.strip().str.title()
print("Unique Region values after fix:", df['Region'].unique())

# ── 2(c) Parse mixed date formats ──
def parse_mixed_date(date_str):
    """Try multiple date formats and return a datetime object."""
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y'):
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    return pd.NaT  # return Not-a-Time if all formats fail

df['Order_Date'] = df['Order_Date'].apply(parse_mixed_date)
df['Quarter'] = df['Order_Date'].dt.to_period('Q').astype(str)   # e.g. "2024Q1"

print(f"\nDate parse failures (NaT): {df['Order_Date'].isnull().sum()}")
print(f"\nSample parsed dates and quarters:")
print(df[['Order_Date','Quarter']].sample(6, random_state=1))

In [ ]:
# ── Final cleaned dataset ──
print("CLEANED DATASET SUMMARY")
print("=" * 40)
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Date range: {df['Order_Date'].min().date()} → {df['Order_Date'].max().date()}")
print(f"Regions: {sorted(df['Region'].unique())}")
print(f"\nSales_Amount stats after cleaning:")
print(df['Sales_Amount'].describe().round(2))
df.head(8)

## Task 3 — Transform for BI Reporting

Aggregate the cleaned data to **one row per Region × Quarter × Category** with summed `Sales_Amount`.  
This is the standard granularity for BI dashboards (e.g., Power BI / Tableau pivot tables).


In [ ]:
# ── Pivot / aggregate to BI reporting shape ──
bi_df = (
    df.groupby(['Region', 'Quarter', 'Category'], as_index=False)
    .agg(Total_Sales=('Sales_Amount', 'sum'),
         Order_Count=('Order_ID', 'count'))
    .sort_values(['Region', 'Quarter', 'Category'])
    .reset_index(drop=True)
)

bi_df['Total_Sales'] = bi_df['Total_Sales'].round(2)

print(f"BI table shape: {bi_df.shape}")
print(f"\nRows per region:")
print(bi_df.groupby('Region').size())
bi_df.head(16)

In [ ]:
# ── Quick visual: Total Sales by Region & Quarter ──
pivot = bi_df.pivot_table(
    index='Quarter', columns='Region', values='Total_Sales', aggfunc='sum', fill_value=0
)

pivot.plot(kind='bar', figsize=(12, 5), colormap='Set2', width=0.7)
plt.title('Total Sales by Region and Quarter (2024)', fontsize=13, fontweight='bold')
plt.xlabel('Quarter')
plt.ylabel('Total Sales (₹)')
plt.xticks(rotation=0)
plt.legend(title='Region', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## Task 4 — Analytic Function Equivalents in Pandas

We implement three SQL window-function equivalents, all **grouped by Region**.

| SQL Window Function | Pandas Equivalent |
|---|---|
| `RANK() OVER (PARTITION BY Region ORDER BY Total_Sales DESC)` | `.rank(ascending=False, method='min')` within group |
| `LAG(Total_Sales) OVER (PARTITION BY Region ORDER BY Quarter)` | `.shift(1)` on sorted group |
| `SUM(Total_Sales) OVER (PARTITION BY Region ORDER BY Quarter ROWS UNBOUNDED)` | `.cumsum()` on sorted group |

We work on a **Region × Quarter** aggregate (summed across Categories) for cleaner window calculations.


In [ ]:
# ── Region × Quarter aggregate (base for all analytic functions) ──
region_quarter = (
    bi_df.groupby(['Region', 'Quarter'], as_index=False)
    .agg(Total_Sales=('Total_Sales', 'sum'))
    .sort_values(['Region', 'Quarter'])
    .reset_index(drop=True)
)

print("Region × Quarter aggregate:")
region_quarter

### 4(a) Rank — Sales rank within each Region (across quarters)

**SQL equivalent:**
```sql
RANK() OVER (PARTITION BY Region ORDER BY Total_Sales DESC)
```


In [ ]:
# ── 4(a) Rank within Region by Total_Sales (descending) ──
region_quarter['Sales_Rank'] = (
    region_quarter
    .groupby('Region')['Total_Sales']
    .rank(ascending=False, method='min')   # 'min' = tied rows get the lower rank (SQL RANK behaviour)
    .astype(int)
)

print("Rank of each quarter's sales within its Region:")
print(region_quarter[['Region','Quarter','Total_Sales','Sales_Rank']].to_string(index=False))

### 4(b) Period-over-Period Comparison — Previous Quarter's Sales (LAG)

**SQL equivalent:**
```sql
LAG(Total_Sales, 1) OVER (PARTITION BY Region ORDER BY Quarter)
```


In [ ]:
# ── 4(b) LAG equivalent: previous quarter sales per Region ──
region_quarter['Prev_Quarter_Sales'] = (
    region_quarter
    .groupby('Region')['Total_Sales']
    .shift(1)           # shift(1) = one period back = LAG(1)
)

# Derived: quarter-over-quarter growth
region_quarter['QoQ_Growth_pct'] = (
    (region_quarter['Total_Sales'] - region_quarter['Prev_Quarter_Sales'])
    / region_quarter['Prev_Quarter_Sales'] * 100
).round(1)

print("Period-over-Period comparison (LAG):")
cols = ['Region','Quarter','Total_Sales','Prev_Quarter_Sales','QoQ_Growth_pct']
print(region_quarter[cols].to_string(index=False))

### 4(c) Running Total — Cumulative Sales per Region

**SQL equivalent:**
```sql
SUM(Total_Sales) OVER (PARTITION BY Region ORDER BY Quarter
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
```


In [ ]:
# ── 4(c) Running total (CUMSUM) per Region, ordered by Quarter ──
region_quarter['Running_Total'] = (
    region_quarter
    .groupby('Region')['Total_Sales']
    .cumsum()         # cumulative sum within each group, in existing sort order
)

print("Running Total of Sales per Region:")
cols = ['Region','Quarter','Total_Sales','Running_Total']
print(region_quarter[cols].to_string(index=False))

In [ ]:
# ── Final analytic table ──
print("\nCOMPLETE ANALYTIC TABLE")
print("=" * 75)
print(region_quarter.to_string(index=False))

In [ ]:
# ── Running Total visualization ──
fig, ax = plt.subplots(figsize=(12, 5))

for region, grp in region_quarter.groupby('Region'):
    ax.plot(grp['Quarter'], grp['Running_Total'], marker='o', label=region, linewidth=2)

ax.set_title('Running Total of Sales by Region (2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Quarter')
ax.set_ylabel('Cumulative Sales (₹)')
ax.legend(title='Region')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Task 5 — Gemini's Explanation of the GroupBy / Window Code Block

I asked Gemini to explain the following code block in its own words:

```python
region_quarter['Running_Total'] = (
    region_quarter
    .groupby('Region')['Total_Sales']
    .cumsum()
)
```

---

### Gemini's Explanation (pasted verbatim from the chat panel):

> *"This code calculates a running total of `Total_Sales` for each `Region` independently.  
> First, `groupby('Region')` splits the DataFrame into separate groups — one for each unique region (North, South, East, West).  
> Then, `['Total_Sales']` selects only the sales column within those groups.  
> Finally, `.cumsum()` computes a cumulative sum within each group, meaning each row's value is the sum of all `Total_Sales` values from the first row of that region up to and including the current row.  
> The result is stored back in a new column `Running_Total`. Since the data is already sorted by Quarter before this step, the running total naturally progresses from Q1 through Q4 for each region."*

---

### My Accuracy Critique:

**Mostly accurate — with one important gap.**

✅ Gemini correctly described `groupby` as partitioning by Region, and `cumsum` as accumulating within each partition.  
✅ It correctly noted that the ordering of the result depends on the sort order already present in the DataFrame — which is why the `.sort_values(['Region', 'Quarter'])` step earlier in Task 4 matters.

❌ **However**, Gemini did not mention that `groupby()` + `cumsum()` resets the cumulative count at the start of each new group. A student who hadn't tested this might assume `cumsum()` is global (continuing across regions), which would produce wrong results. I had to verify this by inspecting the output and confirming the running total restarts at each new Region.

❌ Gemini also omitted that this is the Pandas equivalent of a SQL `SUM(...) OVER (PARTITION BY Region ORDER BY Quarter ROWS UNBOUNDED PRECEDING)` window function — the SQL parallel is what makes this analytically meaningful for a BI context.

**Verdict:** Gemini's explanation is a good starting point for a beginner but lacks precision about group-reset behavior and the SQL analogy. It needed a correction note before being accepted.


## Reflection — Gemini-Assisted vs. Self-Corrected Work



Working with Gemini as an in-editor assistant revealed a clear pattern: it excels at syntax recall but requires critical oversight for analytical decisions.

**Where Gemini was directly usable:**  
The date-parsing skeleton — using `pd.to_datetime` with a format string — was suggested correctly and worked with minor adaptation. Similarly, the `.groupby().rank()` and `.cumsum()` syntax was accurate on the first try. For boilerplate tasks like sorting, renaming columns, and building the bar chart, Gemini's autocomplete saved real time.

**Where I had to fix or reject suggestions:**  
The most significant correction was in missing-value handling. Gemini proposed global mean imputation (`fillna(mean())`), which is naïve: the global mean was inflated by the ₹98,000 outlier and ignored the Region × Category context. I replaced it with grouped-median imputation, which is far more defensible. For the outlier, Gemini suggested simply dropping the row — I chose Winsorization (IQR capping) instead, since dropping an order loses valid transactional context.

The key takeaway: Gemini is a strong *syntax assistant* but a weak *statistical reasoner*. Every analytical decision still required domain judgment and deliberate review.
